# Lab Assignment 13

**Prerequisite:** the class demo notebook *ICL_and_Instruction_Finetuning_Demo.ipynb*  
**Runtime:** Google Colab with a T4 GPU (about 10–15 minutes of GPU time per task)

Each task is a small, real NLP problem: you design the solution, run the experiments and explain what you found. Only data loading and evaluation functions are given, so that everyone's numbers are comparable.

## Rules
- Use only the models named in each task.
- You may reuse any code from the class demo. Cite any other source.
- Fix random seeds and report the settings you used.
- The two tasks are independent, so you can restart the runtime between them.

## Submission
Submit this notebook as `.ipynb` **with all outputs visible**.

## 0. Setup

In [ ]:
!pip install -q transformers datasets accelerate sentence-transformers scikit-learn

In [ ]:
import re, time, random, gc, sqlite3, warnings
from collections import Counter
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq,
                          Trainer, TrainingArguments)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 300)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_causal_lm(name):
    """Load a decoder-only model in fp32 (numerically safe on a T4) and its tokenizer."""
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name).to(device).eval()
    return tok, model


@torch.no_grad()
def generate_batch(model, tok, prompts, max_new_tokens=64, batch_size=8, stop_strings=None, desc="generating"):
    """Greedy, batched generation. Returns only the new text for each prompt, cut at the first stop string."""
    kwargs = dict(stop_strings=stop_strings, tokenizer=tok) if stop_strings else {}
    old_side, tok.padding_side = tok.padding_side, "left"
    outputs = []
    for i in tqdm(range(0, len(prompts), batch_size), desc=desc, leave=False):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id, **kwargs)
        for row in gen[:, enc["input_ids"].shape[1]:]:
            text = tok.decode(row, skip_special_tokens=True)
            for s in (stop_strings or []):
                text = text.split(s)[0]
            outputs.append(text.strip())
    tok.padding_side = old_side
    return outputs

---
# Section A: In-Context Learning

## Task 1. A news-topic classifier with zero training (50 marks)

**Scenario.** A news website wants to tag articles as **World**, **Sports**, **Business** or **Sci/Tech**. It has only 200 labelled articles and no budget to train a model. Build the best classifier you can using **only in-context learning**, then decide whether it is actually a better choice than a simple classical model.

**Constraints**
- Model: `Qwen/Qwen2.5-0.5B` (base). No weight updates.
- Demonstrations may only come from `ag_pool`, with **at most 8 per prompt**.
- Every prediction must be one of the four classes. Report final numbers on `ag_test` using `report_classification`.

**What you must do**
1. **Baselines.** Evaluate zero-shot and 8 randomly chosen demonstrations.
2. **Retrieval-based demonstrations.** For each test article, choose the 8 most similar articles from `ag_pool` as demonstrations, using sentence embeddings from `sentence-transformers/all-MiniLM-L6-v2`. Compare with random selection.
3. **Classical baseline.** Train TF-IDF + logistic regression on the same 200 articles.
4. **Error analysis.** Show the confusion matrix of your best system and explain 3 of its mistakes.

**Analysis questions** (about 250 words in total)
- (a) Did retrieval-based demonstrations beat random ones? Why do you think similar examples help?
- (b) Which classes get confused most often, and why might that be?
- (c) Compare your best ICL system with logistic regression on accuracy **and** speed (seconds per article). Which would you deploy for this website, and why?



In [ ]:
ag = load_dataset("fancyzhx/ag_news")
AG_LABELS = ["World", "Sports", "Business", "Sci/Tech"]

ag_test = ag["test"].shuffle(seed=0).select(range(200))
_train = ag["train"].shuffle(seed=1)
ag_pool = _train.select(range(200))          # the ONLY labelled data you may use
ag_dev = _train.select(range(200, 300))      # use for design decisions (prompt format, label words)

for name, ds in [("pool", ag_pool), ("dev", ag_dev), ("test", ag_test)]:
    counts = Counter(ds["label"])
    print(f"{name:5s}", {AG_LABELS[k]: counts[k] for k in range(4)})
print("\nExample:", ag_pool[0])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

RESULTS_T1 = []   # append the dict returned by report_classification for every system


def report_classification(y_true, y_pred, method, sec_per_example=None, show_cm=False):
    """Fixed evaluation for Task 1. y_true / y_pred are lists of label ids (0-3)."""
    res = {"method": method,
           "accuracy": accuracy_score(y_true, y_pred),
           "macro_f1": f1_score(y_true, y_pred, average="macro"),
           "sec_per_example": sec_per_example}
    if show_cm:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=list(range(4)), display_labels=AG_LABELS,
                                                cmap="Blues", xticks_rotation=45)
        plt.title(method); plt.show()
    return res

### Task 1: Your solution
Add as many code and text cells as you need.

**Settings used (reproducibility).** Model `Qwen/Qwen2.5-0.5B` (base, fp32, no weight updates), greedy scoring, seeds fixed with `set_seed(0)`; random demonstrations use seeds 0, 1, 2. Every prediction is the arg-max over exactly four label words, so it is always one of the four classes.

**Method.** Instead of letting the model generate free text, each prompt ends with `Topic:` and we compare the log-probability that the model assigns to the first token of each label word (` World`, ` Sports`, ` Business`, ` Sci`/` Technology`/` Science`). This guarantees a valid class, is deterministic, and needs a single forward pass per article. The prompt format and the label words were chosen on `ag_dev` only; `ag_test` is used once for the final numbers.

In [ ]:
# ---- 1. Load the models and define the helpers ----
from IPython.display import Markdown
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def show_md(text):
    display(Markdown(text))

set_seed(0)
tok_q, model_q = load_causal_lm("Qwen/Qwen2.5-0.5B")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

pool_texts, pool_labels = list(ag_pool["text"]), list(ag_pool["label"])
dev_texts,  dev_labels  = list(ag_dev["text"]),  list(ag_dev["label"])
test_texts, test_labels = list(ag_test["text"]), list(ag_test["label"])

LABEL_SETS = {
    "Sci/Tech":   ["World", "Sports", "Business", "Sci/Tech"],
    "Technology": ["World", "Sports", "Business", "Technology"],
    "Science":    ["World", "Sports", "Business", "Science"],
}

def fmt_article(t, n=400):
    """One line, at most n characters (keeps 8-shot prompts short enough for a T4)."""
    return " ".join(t.split())[:n]

def build_prompt(article, demos, words):
    head = f"Classify each news article into one of four topics: {', '.join(words)}.\n\n"
    shots = "".join(f"Article: {fmt_article(t)}\nTopic: {words[l]}\n\n" for t, l in demos)
    return head + shots + f"Article: {fmt_article(article)}\nTopic:"

@torch.no_grad()
def label_scores(model, tok, prompts, words, batch_size=8):
    """Log-probability of the first token of every label word after the prompt -> array (N, 4)."""
    first_ids = [tok(" " + w, add_special_tokens=False).input_ids[0] for w in words]
    if len(set(first_ids)) < len(words):
        raise ValueError(f"label words {words} share their first token; pick other words")
    old_side, tok.padding_side = tok.padding_side, "left"
    out = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="scoring", leave=False):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        hidden = model.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state[:, -1]
        logp = torch.log_softmax(model.lm_head(hidden).float(), dim=-1)      # only the last position -> small memory
        out.append(logp[:, first_ids].cpu())
    tok.padding_side = old_side
    return torch.cat(out).numpy()

pool_emb = embedder.encode(pool_texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

def random_demo_ids(i, k, seed):
    rng = np.random.RandomState(seed * 100003 + i)
    return [int(j) for j in rng.choice(len(pool_texts), k, replace=False)]

def run_system(texts, words, mode, k=8, seed=0, order="similar_last"):
    """mode: 'zero' | 'random' | 'retrieval'. Returns predictions, label scores, seconds per article, demo ids used."""
    t0 = time.perf_counter()
    if mode == "zero":
        demos = [[] for _ in texts]
    elif mode == "random":
        demos = [random_demo_ids(i, k, seed) for i in range(len(texts))]
    elif mode == "retrieval":                      # the 8 most similar pool articles (MiniLM cosine similarity)
        q = embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        demos = []
        for row in q @ pool_emb.T:
            top = [int(j) for j in np.argsort(-row)[:k]]
            demos.append(top[::-1] if order == "similar_last" else top)   # similar_last: closest example right before the query
    else:
        raise ValueError(mode)
    prompts = [build_prompt(t, [(pool_texts[j], pool_labels[j]) for j in d], words) for t, d in zip(texts, demos)]
    scores = label_scores(model_q, tok_q, prompts, words)
    return dict(preds=scores.argmax(1), scores=scores, sec=(time.perf_counter() - t0) / len(texts), demos=demos, prompts=prompts)

acc = lambda y, p: float(np.mean(np.asarray(y) == np.asarray(p)))
print("models and helpers ready; example prompt:\n")
print(build_prompt(test_texts[0], [(pool_texts[0], pool_labels[0])], LABEL_SETS["Sci/Tech"]))

#### 1.1 Design decisions on `ag_dev` (never on the test set)
Two choices are made on the 100 dev articles: **(i)** which word to use for class 3 (`Sci/Tech`, `Technology` or `Science`, with 8 random demonstrations), and **(ii)** the order of the retrieved demonstrations (most similar example **last**, i.e. closest to the query, or **first**).

In [ ]:
# (i) label word for class 3, evaluated with 8 random demonstrations (seed 0)
dev_rows = []
for name, words in LABEL_SETS.items():
    r = run_system(dev_texts, words, "random", seed=0)
    dev_rows.append({"label word for class 3": name, "dev accuracy": acc(dev_labels, r["preds"])})
df_dev_words = pd.DataFrame(dev_rows); display(df_dev_words)
BEST_SET = df_dev_words.sort_values("dev accuracy", ascending=False).iloc[0]["label word for class 3"]
WORDS = LABEL_SETS[BEST_SET]
print("chosen label words:", WORDS)

# (ii) order of the retrieved demonstrations
order_rows = []
for order in ["similar_last", "similar_first"]:
    r = run_system(dev_texts, WORDS, "retrieval", order=order)
    order_rows.append({"order of retrieved demos": order, "dev accuracy": acc(dev_labels, r["preds"])})
df_dev_order = pd.DataFrame(order_rows); display(df_dev_order)
ORDER = df_dev_order.sort_values("dev accuracy", ascending=False).iloc[0]["order of retrieved demos"]
print("chosen order:", ORDER)

#### 1.2 The four systems on `ag_test` (zero-shot, random 8-shot, retrieval 8-shot, TF-IDF + logistic regression)

In [ ]:
SYS = {}          # name -> dict(preds, scores, sec, demos)
RESULTS_T1.clear()

def register(name, res, show_cm=False):
    SYS[name] = res
    RESULTS_T1.append(report_classification(test_labels, res["preds"], name, res["sec"], show_cm=show_cm))

# 1. zero-shot
register("ICL zero-shot", run_system(test_texts, WORDS, "zero"))
# 2. 8 random demonstrations (three seeds, so we can see how much the random choice matters)
for s in [0, 1, 2]:
    register(f"ICL random 8-shot (seed {s})", run_system(test_texts, WORDS, "random", seed=s))
# 3. 8 retrieved demonstrations (all-MiniLM-L6-v2)
register("ICL retrieval 8-shot", run_system(test_texts, WORDS, "retrieval", order=ORDER))

# 4. classical baseline: TF-IDF + logistic regression trained on the same 200 articles (C chosen on dev)
def make_lr(C):
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english"),
                         LogisticRegression(C=C, max_iter=2000))
dev_c = {C: acc(dev_labels, make_lr(C).fit(pool_texts, pool_labels).predict(dev_texts)) for C in [1, 10, 100]}
BEST_C = max(dev_c, key=dev_c.get)
print("logistic regression: dev accuracy for C =", dev_c, "-> C =", BEST_C)
t0 = time.perf_counter(); lr = make_lr(BEST_C).fit(pool_texts, pool_labels); lr_fit_sec = time.perf_counter() - t0
t0 = time.perf_counter()
for _ in range(20):
    lr_preds = lr.predict(test_texts)
lr_sec = (time.perf_counter() - t0) / 20 / len(test_texts)
register("TF-IDF + logistic regression", dict(preds=np.asarray(lr_preds), scores=None, sec=lr_sec, demos=None))
print(f"logistic regression: fit {lr_fit_sec:.2f} s on 200 articles, prediction {1000 * lr_sec:.3f} ms per article")

### Task 1: Results and analysis ✍️

_Results table, plots, and answers to the analysis questions go here._

In [ ]:
df_t1 = pd.DataFrame(RESULTS_T1).set_index("method")
display(df_t1.style.format({"accuracy": "{:.3f}", "macro_f1": "{:.3f}", "sec_per_example": "{:.4f}"}))

rand_names = [f"ICL random 8-shot (seed {s})" for s in [0, 1, 2]]
rand_acc = df_t1.loc[rand_names, "accuracy"].values
print(f"random 8-shot accuracy over 3 seeds: {rand_acc.mean():.3f} +/- {rand_acc.std():.3f}   (retrieval 8-shot: {df_t1.loc['ICL retrieval 8-shot', 'accuracy']:.3f})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
shown = df_t1.copy(); colors = ["tab:gray", "tab:blue", "tab:blue", "tab:blue", "tab:green", "tab:orange"]
ax[0].barh(shown.index, shown["accuracy"], color=colors); ax[0].set_xlim(0, 1); ax[0].set_xlabel("accuracy on ag_test"); ax[0].invert_yaxis()
ax[1].barh(shown.index, shown["sec_per_example"], color=colors); ax[1].set_xscale("log"); ax[1].set_xlabel("seconds per article (log scale)"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

#### 1.3 Error analysis: confusion matrix of the best ICL system and three mistakes
The "best system" below is the best **in-context-learning** system on `ag_test` (accuracy; ties broken by macro-F1). The confusion matrix of logistic regression is shown for comparison.

In [ ]:
icl_names = ["ICL zero-shot", "ICL retrieval 8-shot"] + rand_names
BEST_ICL = max(icl_names, key=lambda n: (df_t1.loc[n, "accuracy"], df_t1.loc[n, "macro_f1"]))
print("best ICL system:", BEST_ICL, "| accuracy", round(df_t1.loc[BEST_ICL, "accuracy"], 3))
_ = report_classification(test_labels, SYS[BEST_ICL]["preds"], f"{BEST_ICL}  (best ICL system)", SYS[BEST_ICL]["sec"], show_cm=True)
_ = report_classification(test_labels, SYS["TF-IDF + logistic regression"]["preds"], "TF-IDF + logistic regression", lr_sec, show_cm=True)

In [ ]:
# Explain three mistakes of the best ICL system: one example from each of the three most frequent confusions
PAIR_NOTE = {
    frozenset([2, 3]): "Business and Sci/Tech overlap: news about technology companies (earnings, mergers, product launches) is about business and about technology at the same time, so the label is genuinely ambiguous.",
    frozenset([0, 2]): "World and Business overlap: economic news about other countries (oil prices, trade, foreign companies) can be filed as world news or as business news.",
    frozenset([0, 3]): "World and Sci/Tech overlap: stories about government policy on science, the internet, security or space involve both topics.",
    frozenset([0, 1]): "World and Sports overlap: sport events tied to countries, politics or security read like world news.",
    frozenset([1, 2]): "Sports and Business overlap: money-related sports stories (contracts, ticket sales, club ownership) use business vocabulary.",
    frozenset([1, 3]): "Sports and Sci/Tech overlap: video games, sports technology or doping science use technical vocabulary.",
}
res = SYS[BEST_ICL]; y = np.asarray(test_labels); pred = res["preds"]
errors = np.where(y != pred)[0]
pair_counts = Counter((int(y[i]), int(pred[i])) for i in errors)
chosen, used_pairs = [], set()
for (g, p), _n in pair_counts.most_common():          # one example per distinct confusion pair, most frequent first
    if len(chosen) == 3: break
    i = next(i for i in errors if (y[i], pred[i]) == (g, p) and i not in chosen)
    chosen.append(int(i)); used_pairs.add((g, p))
for i in errors:                                       # fill up if there were fewer than 3 different pairs
    if len(chosen) == 3: break
    if int(i) not in chosen: chosen.append(int(i))

lr_pred_all = SYS["TF-IDF + logistic regression"]["preds"]
for n, i in enumerate(chosen, 1):
    g, p = int(y[i]), int(pred[i]); sc = res["scores"][i]; prob = np.exp(sc - np.logaddexp.reduce(sc))
    demo_labels = [pool_labels[j] for j in res["demos"][i]] if res["demos"] and res["demos"][i] else []
    n_gold = demo_labels.count(g); n_pred = demo_labels.count(p)
    why = [PAIR_NOTE.get(frozenset([g, p]), "The article contains vocabulary typical of the predicted class.")]
    why.append(f"The model's own probabilities over the four classes were { {k: round(float(v), 2) for k, v in zip(AG_LABELS, prob)} }, so the margin between its answer and the true class was {prob[p] - prob[g]:.2f}.")
    if demo_labels:
        why.append(f"Among the 8 demonstrations in its prompt {n_pred} had the predicted label ({AG_LABELS[p]}) and {n_gold} had the true label ({AG_LABELS[g]})"
                   + (" - the prompt itself pushed the model towards the wrong class." if n_pred > n_gold else " - the demonstrations did not explain the mistake; the model decided against them."))
    why.append("TF-IDF + logistic regression " + ("made the same mistake." if int(lr_pred_all[i]) == p else ("got this article right." if int(lr_pred_all[i]) == g else "predicted a third class.")))
    show_md(f"**Mistake {n}** (test article #{i}) - true: **{AG_LABELS[g]}**, predicted: **{AG_LABELS[p]}**\n\n> {fmt_article(test_texts[i], 350)}\n\n" + "\n".join(f"- {w}" for w in why))

### Task 1: Analysis questions (answers are generated from the results above)
Each answer below is produced by the next cell from the numbers in `RESULTS_T1`, the confusion matrices and the demonstrations that were actually used, so the text always agrees with the tables.

In [ ]:
# ---- (a) Did retrieval-based demonstrations beat random ones? ----
ret_acc, ret_f1 = df_t1.loc["ICL retrieval 8-shot", ["accuracy", "macro_f1"]]
zs_acc = df_t1.loc["ICL zero-shot", "accuracy"]
gap = ret_acc - rand_acc.mean()
if ret_acc > rand_acc.max():      verdict = "**Yes.**"
elif ret_acc > rand_acc.mean():   verdict = "**Slightly, but not beyond the spread between random seeds.**"
else:                             verdict = "**No.**"

def has_gold(sys_name):
    return float(np.mean([int(y[i]) in [pool_labels[j] for j in SYS[sys_name]["demos"][i]] for i in range(len(y))]))
cov_rand = float(np.mean([has_gold(n) for n in rand_names])); cov_ret = has_gold("ICL retrieval 8-shot")
knn_vote = np.array([Counter(pool_labels[j] for j in SYS["ICL retrieval 8-shot"]["demos"][i]).most_common(1)[0][0] for i in range(len(y))])

show_md(f"""**(a) Retrieval vs. random demonstrations.** {verdict} Zero-shot reaches {zs_acc:.3f} accuracy, 8 random demonstrations {rand_acc.mean():.3f} (+/- {rand_acc.std():.3f} over three seeds), and 8 retrieved demonstrations {ret_acc:.3f} (macro-F1 {ret_f1:.3f}), a difference of {100 * gap:+.1f} points to the random mean.

*Why similar examples help (and what we measured).* (1) **Label coverage:** a random set of 8 articles from a 200-article pool contained at least one article of the correct class for {100 * cov_rand:.1f}% of the test articles, the retrieved set for {100 * cov_ret:.1f}%; the model can only copy a label it has seen. (2) **Near-duplicates and shared vocabulary:** similar articles share entities and topic words, so the label of the closest demonstration is strongly predictive. Simply taking the majority label among the 8 retrieved neighbours (a k-NN vote with no language model at all) already gives {acc(test_labels, knn_vote):.3f} accuracy. (3) **Copying mechanism:** as in the class demo (induction heads), the model continues the pattern `Article: ... Topic: <label>` of the demonstrations that look most like the query, and the most similar one sits right before the query (order chosen on dev: *{ORDER}*).""")

# ---- (b) Which classes get confused most often? ----
def top_pairs(pred_vec, n=3):
    c = Counter((int(a), int(b)) for a, b in zip(y, pred_vec) if a != b)
    return c.most_common(n)
tp_icl, tp_lr = top_pairs(pred), top_pairs(lr_pred_all)
fmt_pairs = lambda tp: "; ".join(f"{AG_LABELS[g]} -> {AG_LABELS[p]} ({n} articles)" for (g, p), n in tp)
(g1, p1), n1 = tp_icl[0]
recall = {AG_LABELS[c]: float(np.mean(pred[y == c] == c)) for c in range(4)}
worst_class = min(recall, key=recall.get)
show_md(f"""**(b) Most confused classes.** For the best ICL system ({BEST_ICL}) the most frequent confusions are: {fmt_pairs(tp_icl)}. For logistic regression: {fmt_pairs(tp_lr)}. The class with the lowest recall is **{worst_class}** ({100 * recall[worst_class]:.1f}%); per-class recall: {', '.join(f'{k} {100 * v:.1f}%' for k, v in recall.items())}.

*Why:* {PAIR_NOTE.get(frozenset([g1, p1]), '')} AG News labels come from the news sources' own category tags rather than from a careful reading of every article, so many articles legitimately sit on the border between two topics; no classifier can fully resolve this, and the same kinds of confusion are expected for the classical model (compare the pairs listed above).""")

# ---- (c) ICL vs. logistic regression: accuracy and speed ----
icl_sec = df_t1.loc[BEST_ICL, "sec_per_example"]; lr_acc = df_t1.loc["TF-IDF + logistic regression", "accuracy"]
icl_acc = df_t1.loc[BEST_ICL, "accuracy"]; ratio = icl_sec / lr_sec
if lr_acc > icl_acc + 0.02:
    rec = (f"**I would deploy logistic regression.** It is more accurate than the best ICL system ({lr_acc:.3f} vs {icl_acc:.3f}) and about {ratio:,.0f} times faster, needs no GPU and is trivial to retrain, so ICL has no advantage here.")
elif lr_acc >= icl_acc - 0.02:
    rec = (f"**I would deploy logistic regression.** It is as accurate as the best ICL system (within 2 points: {lr_acc:.3f} vs {icl_acc:.3f}) while being about {ratio:,.0f} times faster, needs no GPU and is trivial to retrain.")
elif icl_acc - lr_acc < 0.05:
    rec = (f"**I would still lean towards logistic regression:** ICL is only {100 * (icl_acc - lr_acc):.1f} points more accurate ({icl_acc:.3f} vs {lr_acc:.3f}) but about {ratio:,.0f} times slower and needs a GPU. If the website cannot get more labelled data and the 2-3 point gain matters, ICL is the alternative.")
else:
    rec = (f"**I would deploy the retrieval-based ICL system if latency and GPU cost are acceptable:** it is {100 * (icl_acc - lr_acc):.1f} points more accurate than logistic regression ({icl_acc:.3f} vs {lr_acc:.3f}) when only 200 labelled articles exist, but it is about {ratio:,.0f} times slower. If the volume of articles is large, logistic regression is the cheap fallback, or a hybrid (logistic regression first, ICL only for low-confidence articles).")
show_md(f"""**(c) ICL vs. logistic regression.** Accuracy: {BEST_ICL} **{icl_acc:.3f}**, TF-IDF + logistic regression **{lr_acc:.3f}** (macro-F1 {df_t1.loc[BEST_ICL, 'macro_f1']:.3f} vs {df_t1.loc['TF-IDF + logistic regression', 'macro_f1']:.3f}). Speed: {icl_sec:.4f} s per article for ICL (batched on a T4 GPU, prompts of about 1,000 tokens) vs {lr_sec:.6f} s for logistic regression ({1 / lr_sec:,.0f} articles per second on the CPU).

{rec} Other factors: ICL needs a 0.5B-parameter model in GPU memory and long prompts at every call, and its output depends on the demonstrations and the prompt wording; logistic regression is small, deterministic and inspectable. ICL's real advantage is flexibility: a new class or a new label scheme needs only new demonstrations, not a new training run.""")

In [ ]:
# Free the GPU before Task 2
del model_q, embedder; clear_gpu()
print("Task 1 finished; GPU memory released.")

---
# Section B: Instruction Fine-Tuning

## Task 2. English → SQL: prompt a ready-made model or fine-tune your own? (50 marks)

**Scenario.** A company wants its analysts to query databases in plain English. Given a table schema (a `CREATE TABLE` statement) and a question, a model must write the SQL query. Should the company **prompt** an existing instruction-tuned model, or **fine-tune** a small model on examples? Build both and recommend one.

**Data:** each example has a `question`, a `context` (the schema) and an `answer` (the gold SQL).

**Constraints**
- Prompting system: `Qwen/Qwen2.5-0.5B-Instruct` (use its chat template).
- Fine-tuned system: `openai-community/gpt2` (124M), with the loss computed on the SQL tokens only.
- Training and demonstration data come only from `sql_train_pool`. Evaluate every system on `sql_test` with `evaluate_sql`.

**What you must do**
1. **Prompting baseline.** Prompt Qwen2.5-0.5B-Instruct with 3 examples from the pool. The model may add explanations or code fences, so extract just the SQL from its output.
2. **Fine-tuning.** Design an input format for (schema, question) → SQL and fine-tune GPT-2 twice, on **500** and on **2,000** examples, with the same hyperparameters.
3. **Error analysis.** Look at 10 mistakes of your best system and label each one: wrong column, wrong condition or aggregate, wrong value, invalid SQL, or correct but written differently.

**Analysis questions** (about 250 words in total)
- (a) How did fine-tuned GPT-2 compare with prompting a model four times its size? How much did going from 500 to 2,000 examples help?
- (b) How many of your "mistakes" were actually correct queries written differently? Why does exact match underestimate performance?
- (c) Write a short recommendation to the company: prompting or fine-tuning? Consider accuracy, the labelled data needed, cost, and what happens when new tables are added.

**Marking:** prompting baseline 12 · fine-tuning (both sizes) 18 · error analysis 8 · discussion 12

In [ ]:
sql_ds = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=0)
sql_test = sql_ds.select(range(200))
sql_train_pool = sql_ds.select(range(200, 5200))   # training, validation and demonstration data

ex = sql_test[0]
print("SCHEMA  :", ex["context"]); print("QUESTION:", ex["question"]); print("GOLD SQL:", ex["answer"])

In [ ]:
def normalize_sql(q):
    """Lenient normalisation for exact match: case, whitespace, quote style, trailing semicolon."""
    q = q.strip().rstrip(";").replace('"', "'")
    q = re.sub(r"\s+", " ", q)
    q = re.sub(r"\s*([(),=<>])\s*", r"\1", q)
    return q.lower().strip()


def executes(sql, context):
    """True if the query runs without error on an empty database built from the schema."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(context)
        con.execute(sql).fetchall()
        return True
    except Exception:
        return False
    finally:
        con.close()


def evaluate_sql(preds, dataset, system):
    """Fixed evaluation for Task 2. preds: list of SQL strings aligned with dataset."""
    golds, contexts = dataset["answer"], dataset["context"]
    return {"system": system,
            "exact_match": float(np.mean([normalize_sql(p) == normalize_sql(g) for p, g in zip(preds, golds)])),
            "executable": float(np.mean([executes(p, c) for p, c in zip(preds, contexts)]))}


RESULTS_T2 = []   # append the dict returned by evaluate_sql for every system
print("Reference (gold queries):", evaluate_sql(sql_test["answer"], sql_test, "gold"))

### Task 2: Your solution
Add as many code and text cells as you need.

**Settings used (reproducibility).** Prompting: `Qwen/Qwen2.5-0.5B-Instruct`, chat template, 3 fixed demonstrations from `sql_train_pool` (seed 0), greedy decoding, at most 128 new tokens, repetition penalty switched off (the model's default of 1.1 would corrupt SQL). Fine-tuning: `openai-community/gpt2`, format `### Schema / ### Question / ### SQL`, loss on the SQL tokens and the end-of-text token only, **500** and **2,000** examples (the 500 are a subset of the 2,000), identical hyper-parameters for both: 3 epochs, batch size 16, learning rate 1e-4, cosine schedule with 10 warm-up steps, fp16, seed 42. Examples longer than 256 GPT-2 tokens are excluded from the training data.

In [ ]:
# ---- 2.1 Prompting baseline: Qwen2.5-0.5B-Instruct with 3 demonstrations ----
set_seed(0)
tok_i, model_i = load_causal_lm("Qwen/Qwen2.5-0.5B-Instruct")
model_i.generation_config.repetition_penalty = 1.0       # the default 1.1 penalises repeated column/table names

rng0 = np.random.RandomState(0)
cand = [i for i in rng0.permutation(len(sql_train_pool))[:100] if len(sql_train_pool[int(i)]["context"]) < 300]
FEWSHOT = [sql_train_pool[int(i)] for i in cand[:3]]
print("The 3 demonstrations:")
for s in FEWSHOT:
    print("  ", s["question"], "->", s["answer"])

SYSTEM_MSG = ("You are an expert SQL assistant. Given a table schema (CREATE TABLE statement) and a question, "
              "reply with ONE SQL query that answers the question and nothing else.")

def user_msg(ex):
    return f"Schema:\n{ex['context']}\n\nQuestion: {ex['question']}"

def chat_prompt(ex, shots=FEWSHOT):
    msgs = [{"role": "system", "content": SYSTEM_MSG}]
    for s in shots:
        msgs += [{"role": "user", "content": user_msg(s)}, {"role": "assistant", "content": s["answer"]}]
    msgs.append({"role": "user", "content": user_msg(ex)})
    return tok_i.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def extract_sql(text):
    """Pull just the SQL out of a chat answer: remove ``` fences, explanations and anything after the first statement."""
    m = re.search(r"```(?:sql|SQL)?\s*(.*?)```", text, flags=re.S)
    if m:
        text = m.group(1)
    text = text.replace("```", "")
    m = re.search(r"\b(select|with)\b", text, flags=re.I)
    if m:
        text = text[m.start():]
    text = re.split(r"\n\s*(?:Explanation|Note|This query|The query|In this|Here)", text)[0]
    text = text.split(";")[0]
    text = re.split(r"\n\s*\n", text)[0]
    return " ".join(text.split()).strip()

print("\nExample prompt (last 700 characters):\n", chat_prompt(sql_test[0])[-700:])

In [ ]:
t0 = time.time()
raw_prompting = generate_batch(model_i, tok_i, [chat_prompt(ex) for ex in sql_test], max_new_tokens=128, batch_size=8, desc="Qwen-Instruct")
prompt_preds = [extract_sql(o) for o in raw_prompting]
print(f"generated {len(raw_prompting)} answers in {time.time() - t0:.0f} s")

n_fence = sum("```" in o for o in raw_prompting); n_changed = sum(o.strip() != p for o, p in zip(raw_prompting, prompt_preds))
n_empty = sum(p == "" for p in prompt_preds)
print(f"raw outputs with code fences: {n_fence}/200 | outputs changed by SQL extraction: {n_changed}/200 | empty after extraction: {n_empty}/200")
for o, p, ex in list(zip(raw_prompting, prompt_preds, sql_test))[:3]:
    print("\nRAW      :", repr(o[:200])); print("EXTRACTED:", p); print("GOLD     :", ex["answer"])

RESULTS_T2.clear()
RESULTS_T2.append(evaluate_sql(prompt_preds, sql_test, "Qwen2.5-0.5B-Instruct, 3-shot prompting"))
print("\n", RESULTS_T2[-1])
del model_i; clear_gpu()

#### 2.2 Fine-tuning GPT-2 on 500 and on 2,000 examples

In [ ]:
# ---- Input format and loss masking (loss only on the SQL tokens + end-of-text) ----
tok_gpt = AutoTokenizer.from_pretrained("openai-community/gpt2")
tok_gpt.pad_token = tok_gpt.eos_token
MAX_LEN = 256

def sql_prompt(context, question):
    return f"### Schema:\n{context}\n\n### Question:\n{question}\n\n### SQL:\n"

def tokenize_sql(ex):
    p = tok_gpt(sql_prompt(ex["context"], ex["question"]), add_special_tokens=False).input_ids
    r = tok_gpt(ex["answer"], add_special_tokens=False).input_ids + [tok_gpt.eos_token_id]
    return {"input_ids": p + r, "attention_mask": [1] * (len(p) + len(r)), "labels": [-100] * len(p) + r}

tokenized = sql_train_pool.map(tokenize_sql, remove_columns=sql_train_pool.column_names)
tokenized = tokenized.filter(lambda e: len(e["input_ids"]) <= MAX_LEN)
print(f"{len(tokenized)} of {len(sql_train_pool)} pool examples fit into {MAX_LEN} GPT-2 tokens")
order_idx = np.random.RandomState(42).permutation(len(tokenized))
train_sets = {500: tokenized.select([int(i) for i in order_idx[:500]]),          # the 500 are a subset of the 2,000
              2000: tokenized.select([int(i) for i in order_idx[:2000]])}
val_ds = tokenized.select([int(i) for i in order_idx[-200:]])                      # 200 held-out pool examples (validation loss)
print({k: len(v) for k, v in train_sets.items()}, "| validation:", len(val_ds))
print("\nOne training sequence (prompt part is masked with -100):")
ex0 = train_sets[500][0]
print(repr(tok_gpt.decode(ex0["input_ids"])))
print("trained on:", repr(tok_gpt.decode([t for t, l in zip(ex0["input_ids"], ex0["labels"]) if l != -100])))

In [ ]:
def finetune(n, seed=42):
    set_seed(seed)
    model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(device)
    model.config.pad_token_id = tok_gpt.eos_token_id
    args = TrainingArguments(output_dir=f"./gpt2-sql-{n}", per_device_train_batch_size=16, num_train_epochs=3,
                             learning_rate=1e-4, warmup_steps=10, lr_scheduler_type="cosine", logging_steps=10,
                             save_strategy="no", fp16=(device == "cuda"), report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=train_sets[n], eval_dataset=val_ds,
                      data_collator=DataCollatorForSeq2Seq(tok_gpt, padding=True, label_pad_token_id=-100))
    t0 = time.time(); trainer.train(); train_sec = time.time() - t0
    val_loss = trainer.evaluate()["eval_loss"]
    model.eval()
    return model, [l for l in trainer.state.log_history if "loss" in l], val_loss, train_sec

def clean_gpt_sql(text):
    text = text.split("###")[0]
    text = re.split(r"\n\s*\n", text.strip())[0]
    return " ".join(text.split(";")[0].split()).strip()

FT = {}
for n in [500, 2000]:
    print(f"\n=== fine-tuning GPT-2 on {n} examples ===")
    model, logs, val_loss, train_sec = finetune(n)
    gen_prompts = [sql_prompt(ex["context"], ex["question"]) for ex in sql_test]
    try:                                       # stop at "###" if the installed transformers supports stop strings for this tokenizer
        raw = generate_batch(model, tok_gpt, gen_prompts, max_new_tokens=128, batch_size=16, stop_strings=["###"], desc=f"GPT-2 ft {n}")
    except Exception as err:                   # otherwise rely on the end-of-text token the model was trained to emit
        print("stop_strings not available, generating without them:", type(err).__name__)
        raw = generate_batch(model, tok_gpt, gen_prompts, max_new_tokens=128, batch_size=16, desc=f"GPT-2 ft {n}")
    preds = [clean_gpt_sql(o) for o in raw]
    FT[n] = dict(preds=preds, raw=raw, logs=logs, val_loss=val_loss, train_sec=train_sec)
    RESULTS_T2.append(evaluate_sql(preds, sql_test, f"GPT-2 fine-tuned on {n}"))
    print(RESULTS_T2[-1], f"| validation loss {val_loss:.3f} | training time {train_sec:.0f} s")
    del model; clear_gpu()

In [ ]:
# Training curves for both sizes
plt.figure(figsize=(7, 4))
for n, d in FT.items():
    plt.plot([l["step"] for l in d["logs"]], [l["loss"] for l in d["logs"]], label=f"{n} examples (val loss {d['val_loss']:.2f})")
plt.xlabel("training step"); plt.ylabel("loss on SQL tokens"); plt.title("GPT-2 fine-tuning on English -> SQL"); plt.legend(); plt.show()

### Task 2: Results and analysis ✍️

_Results table, plots, and answers to the analysis questions go here._

#### 2.3 Results table, with an extra execution-based metric
`exact_match` and `executable` come from the fixed `evaluate_sql`. Because exact match punishes correct queries that are written differently, we add one more column, **`execution match` (our own estimate)**: both queries are run on the same small, automatically populated database built from the schema (3 different random fillings, values drawn from the literals in the two queries); the prediction counts as equivalent when it returns exactly the same rows as the gold query and the gold query returns at least one row. It is an estimate, not a proof of equivalence.

In [ ]:
# ---- execution-based equivalence (extra metric) and error categories ----
SQL_KEYWORDS = set("select from where and or not as count sum avg min max group by order limit asc desc distinct like in is null between having join on left right inner outer union all".split())

def _literals(*queries):
    texts, nums = [], []
    for q in queries:
        for a, b in re.findall(r"'([^']*)'|\"([^\"]*)\"", q):
            texts.append(a or b)
        nums += re.findall(r"(?<![\w.])-?\d+(?:\.\d+)?", q)
    return list(dict.fromkeys(texts)), list(dict.fromkeys(nums))

def _make_db(context, lit_text, lit_num, seed):
    con = sqlite3.connect(":memory:"); con.executescript(context)
    rng = random.Random(seed)
    num_pool = [float(x) if "." in x else int(x) for x in lit_num] + list(range(0, 12))
    txt_pool = lit_text + [x for x in lit_num] + ["a", "b", "c", "d"]
    for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall():
        cols = con.execute(f'PRAGMA table_info("{t}")').fetchall()
        for _ in range(25):
            vals = []
            for c in cols:
                ctype = (c[2] or "").upper()
                numeric = any(k in ctype for k in ("INT", "REAL", "NUM", "FLOA", "DOUB", "DEC"))
                vals.append(rng.choice(num_pool) if numeric else rng.choice(txt_pool))
            con.execute(f'INSERT OR IGNORE INTO "{t}" VALUES ({",".join("?" * len(cols))})', vals)
    return con

def execution_match(pred, gold, context):
    """True if pred returns the same rows as gold on 3 populated databases (and gold returns rows on at least one)."""
    if not pred.strip():
        return False
    lit_text, lit_num = _literals(pred, gold)
    gold_nonempty = False
    for seed in range(3):
        try:
            con = _make_db(context, lit_text, lit_num, seed)
            g = sorted(map(repr, con.execute(gold).fetchall()))
            p = sorted(map(repr, con.execute(pred).fetchall()))
            con.close()
        except Exception:
            return False
        if g != p:
            return False
        gold_nonempty |= len(g) > 0
    return gold_nonempty

def _parts(q):
    q = " ".join(q.strip().rstrip(";").split())
    m = re.search(r"select\s+(.*?)\s+from\s+(.*?)(?:\s+where\s+(.*?))?(?:\s+(?:group by|order by|limit)\s+.*)?$", q, flags=re.I | re.S)
    return (m.group(1), m.group(2), m.group(3) or "") if m else None

def _idents(s):
    return {w.lower() for w in re.findall(r"[A-Za-z_][A-Za-z_0-9]*", re.sub(r"'[^']*'|\"[^\"]*\"", " ", s))} - SQL_KEYWORDS

def error_type(pred, gold, context):
    """Automatic label for a wrong prediction: wrong column | wrong condition or aggregate | wrong value | invalid SQL | correct but written differently | other."""
    if not executes(pred, context) or not pred.strip():
        return "invalid SQL"
    if execution_match(pred, gold, context):
        return "correct but written differently"
    pp, gp = _parts(pred), _parts(gold)
    if pp is None or gp is None:
        return "other"
    agg = lambda s: set(re.findall(r"\b(count|sum|avg|min|max)\s*\(", s.lower()))
    if agg(pp[0]) != agg(gp[0]):
        return "wrong condition or aggregate"
    if _idents(pp[0]) != _idents(gp[0]) or _idents(pp[1]) != _idents(gp[1]):
        return "wrong column"
    if bool(pp[2]) != bool(gp[2]):
        return "wrong condition or aggregate"
    if _idents(pp[2]) != _idents(gp[2]):
        return "wrong column"
    ops = lambda s: sorted(re.findall(r"<=|>=|<>|!=|=|<|>|\blike\b|\band\b|\bor\b", re.sub(r"'[^']*'|\"[^\"]*\"", "", s.lower())))
    if ops(pp[2]) != ops(gp[2]):
        return "wrong condition or aggregate"
    lits = lambda s: sorted(a or b or n for a, b, n in re.findall(r"'([^']*)'|\"([^\"]*)\"|(?<![\w.])(-?\d+(?:\.\d+)?)", s))
    if lits(pp[2]) != lits(gp[2]):
        return "wrong value"
    return "other"

golds, contexts = list(sql_test["answer"]), list(sql_test["context"])
SYSTEM_PREDS = {"Qwen2.5-0.5B-Instruct, 3-shot prompting": prompt_preds,
                "GPT-2 fine-tuned on 500": FT[500]["preds"], "GPT-2 fine-tuned on 2000": FT[2000]["preds"]}
for r in RESULTS_T2:
    r["execution match (est.)"] = float(np.mean([execution_match(p, g, c) for p, g, c in zip(SYSTEM_PREDS[r["system"]], golds, contexts)]))
    r["empty outputs"] = int(sum(p == "" for p in SYSTEM_PREDS[r["system"]]))
df_t2 = pd.DataFrame(RESULTS_T2).set_index("system")
display(df_t2.style.format({"exact_match": "{:.3f}", "executable": "{:.3f}", "execution match (est.)": "{:.3f}"}))

fig, ax = plt.subplots(figsize=(8, 4)); w = 0.27; xs = np.arange(len(df_t2))
for k, (col, c) in enumerate([("exact_match", "tab:blue"), ("execution match (est.)", "tab:green"), ("executable", "tab:gray")]):
    ax.bar(xs + (k - 1) * w, df_t2[col], w, label=col, color=c)
ax.set_xticks(xs); ax.set_xticklabels([s.replace(", ", "\n") for s in df_t2.index], fontsize=8); ax.set_ylim(0, 1); ax.legend(); ax.set_title("English -> SQL on sql_test (200 questions)")
plt.tight_layout(); plt.show()

#### 2.4 Error analysis: 10 mistakes of the best system
The best system is the one with the highest exact match (ties broken by `executable`). Each mistake is labelled by the rules in `error_type` (invalid SQL; correct but written differently = same rows on populated databases; otherwise the clause that differs: columns, aggregate/condition, or literal values). **The labels are automatic**, so every query pair is printed in full and can be checked by eye.

In [ ]:
BEST_SQL = max(df_t2.index, key=lambda s: (df_t2.loc[s, "exact_match"], df_t2.loc[s, "executable"]))
best_preds = SYSTEM_PREDS[BEST_SQL]
mistakes = [i for i, (p, g) in enumerate(zip(best_preds, golds)) if normalize_sql(p) != normalize_sql(g)]
all_types = {i: error_type(best_preds[i], golds[i], contexts[i]) for i in mistakes}
print(f"best system: {BEST_SQL} | {len(mistakes)} mistakes (exact match) out of 200")

sample = list(np.random.RandomState(0).choice(mistakes, size=min(10, len(mistakes)), replace=False))
rows = [{"#": k + 1, "question": sql_test[int(i)]["question"], "gold SQL": golds[int(i)], "predicted SQL": best_preds[int(i)], "label (auto)": all_types[int(i)]}
        for k, i in enumerate(sample)]
display(pd.DataFrame(rows).style.set_properties(**{"text-align": "left"}))

cats = ["wrong column", "wrong condition or aggregate", "wrong value", "invalid SQL", "correct but written differently", "other"]
sample_counts = Counter(all_types[int(i)] for i in sample); all_counts = Counter(all_types.values())
display(pd.DataFrame({"10 sampled mistakes": [sample_counts[c] for c in cats],
                      f"all {len(mistakes)} mistakes": [all_counts[c] for c in cats]}, index=cats))

### Task 2: Analysis questions (answers are generated from the results above)

In [ ]:
# ---- (a) fine-tuned GPT-2 vs prompting a 4x larger model; effect of 500 -> 2,000 examples ----
g = lambda s, c: df_t2.loc[s, c]
P, F5, F20 = "Qwen2.5-0.5B-Instruct, 3-shot prompting", "GPT-2 fine-tuned on 500", "GPT-2 fine-tuned on 2000"
best_ft = F20 if g(F20, "exact_match") >= g(F5, "exact_match") else F5
cmp_word = "clearly outperformed" if g(best_ft, "exact_match") - g(P, "exact_match") > 0.05 else ("roughly matched" if abs(g(best_ft, "exact_match") - g(P, "exact_match")) <= 0.05 else "was clearly worse than")
gain = g(F20, "exact_match") - g(F5, "exact_match")
if g(best_ft, "exact_match") > g(P, "exact_match"):
    why_txt = (f"*Why the prompted model falls behind even though it is larger:* it was never trained to produce this exact output style, so it has to infer the conventions of the dataset (how aggregates, quoting and column order are written) from only 3 examples; {n_fence} of its 200 raw answers contained code fences and {n_changed} needed post-processing. The fine-tuned model has seen hundreds or thousands of examples of the exact format and learns it directly, so a smaller model that is specialised can beat a larger general one.")
else:
    why_txt = (f"*Why the prompted model holds up even though GPT-2 was fine-tuned:* Qwen2.5-0.5B-Instruct has already been trained on code and on following instructions, so 3 examples are enough to show it the format ({n_fence} of its 200 raw answers contained code fences and {n_changed} needed post-processing, which our extraction handles), while GPT-2 is a 124M general-text model that must learn SQL from only {len(train_sets[500])} or {len(train_sets[2000])} examples and has limited capacity for it.")
show_md(f"""**(a) Fine-tuned GPT-2 vs. prompting a 4x larger model.** Qwen2.5-0.5B-Instruct (about 500M parameters) prompted with 3 examples reaches an exact match of **{g(P, 'exact_match'):.3f}** ({g(P, 'execution match (est.)'):.3f} by execution match, {g(P, 'executable'):.3f} executable), while GPT-2 (124M parameters) fine-tuned on 500 examples reaches **{g(F5, 'exact_match'):.3f}** and on 2,000 examples **{g(F20, 'exact_match'):.3f}** ({g(F20, 'execution match (est.)'):.3f} by execution match, {g(F20, 'executable'):.3f} executable). So the best fine-tuned small model {cmp_word} the prompted larger model on exact match. Going from 500 to 2,000 training examples changed exact match by **{100 * gain:+.1f} points** (validation loss {FT[500]['val_loss']:.2f} -> {FT[2000]['val_loss']:.2f}), i.e. {'a substantial gain: the model is still data-limited at 500 examples' if gain > 0.05 else ('a small gain: returns are diminishing for this task' if gain > 0.01 else 'no real gain')}.

{why_txt}""")

# ---- (b) how many mistakes were correct queries written differently? ----
n_alt_all = all_counts["correct but written differently"]; n_alt_s = sample_counts["correct but written differently"]
em, ex_m = g(BEST_SQL, "exact_match"), g(BEST_SQL, "execution match (est.)")
show_md(f"""**(b) Mistakes that were actually correct.** Of the 10 sampled mistakes of the best system ({BEST_SQL}), **{n_alt_s}** were labelled *correct but written differently*; over all {len(mistakes)} exact-match mistakes the figure is **{n_alt_all}** ({100 * n_alt_all / max(len(mistakes), 1):.0f}%). The remaining mistakes were: {', '.join(f'{c}: {all_counts[c]}' for c in cats if c != 'correct but written differently')}. Consequently the execution-based estimate ({ex_m:.3f}) is **{100 * (ex_m - em):.1f} points higher** than exact match ({em:.3f}).

*Why exact match underestimates performance:* it compares strings, and one meaning has many spellings in SQL: `COUNT(*)` vs `COUNT(column)`, a different but equivalent order of `AND` conditions, the order of the selected columns, table aliases, `<>` vs `!=`, redundant parentheses or `DISTINCT`, double vs single quotes (partly normalised by `normalize_sql`), `ASC` written explicitly or not. Our check is itself only an estimate (two different queries can coincidentally return the same rows on a small database), so the truth lies between the two metrics.""")

# ---- (c) recommendation ----
ft_better = g(F20, "exact_match") > g(P, "exact_match") + 0.02
rec = ("**Recommendation: fine-tune a small model**, " if ft_better else "**Recommendation: start with prompting**, ")
if ft_better:
    rec += (f"because with 2,000 labelled examples it is more accurate ({g(F20, 'exact_match'):.3f} vs {g(P, 'exact_match'):.3f} exact match; {g(F20, 'execution match (est.)'):.3f} vs {g(P, 'execution match (est.)'):.3f} by execution match) "
            "and it is far cheaper to run: a 124M model with a 60-token prompt and an automatic stop is much cheaper per query than a 3-shot prompt of about 700 tokens.")
else:
    rec += (f"because the prompted model ({g(P, 'exact_match'):.3f} exact match) is competitive with or better than GPT-2 fine-tuned on 2,000 examples ({g(F20, 'exact_match'):.3f}) and needs no labelled data and no training.")
show_md(f"""**(c) Recommendation to the company.** {rec}

*Accuracy:* see the table above (exact match / execution match / executable). *Labelled data:* prompting needs only 3 examples; fine-tuning needs labelled (schema, question, SQL) pairs, and the change from 500 to 2,000 examples was {100 * gain:+.1f} points exact match, {'so quality still grows with the amount of data' if gain > 0.01 else 'so on this task more data brought little extra benefit at this size'}. *Cost:* fine-tuning took {FT[500]['train_sec']:.0f} s and {FT[2000]['train_sec']:.0f} s on a T4 for 500 / 2,000 examples, a one-off cost; prompting has no training cost but pays for long prompts (schema + 3 demonstrations) at every query. *New tables:* the schema is part of the input in both systems, so new tables work without retraining, but a prompted instruction model copes with unseen schemas and new SQL features more robustly because it keeps its general knowledge, whereas the fine-tuned small model can only be trusted on schemas and question types that resemble its training data; if many new kinds of tables appear, the fine-tuning data have to be extended and the model retrained. A practical path is to start with prompting, log the questions and corrected queries analysts accept, and fine-tune once enough labelled pairs have accumulated; always validate generated SQL by executing it (an `executable` rate below 100% means some queries fail outright) and show it to the analyst before it is run on production data.""")

In [ ]:
# Final summary of both tasks
print("TASK 1 (ag_test, 200 articles)"); display(pd.DataFrame(RESULTS_T1).set_index("method").round(4))
print("TASK 2 (sql_test, 200 questions)"); display(df_t2.round(4))